# ESPAC Direct Field Emissions (DFE)

This notebook computes direct field emissions (DFE) from `outputs/CSVs/02_espac_crop_lci_table_filtered__summary_<strategy>.csv` and writes a new file:

- `outputs/CSVs/03-05_espac_crop_lci_table_filtered_dfe__summary_<strategy>.csv`

It follows the DFE formulas in `inputs/00_KNIME_script.txt` (direct emissions section) and loads factors from `inputs/03-05_dfe_factors.yml`, which was generated from `inputs/03_Factors.xlsx`.

## Notes

- The filtered ESPAC LCI table does not contain all KNIME flow variables (for example explicit `N_tot`, `P_tot`, `K_tot`, some fertiliser product composition fields, and `Manure_type`).
- This notebook uses `inputs/03_dfe_nutrient_contents.yml` for mineral + organic nutrient contents and `inputs/03-05_dfe_factors.yml` for emission-model factors.
- The YAML file should be reviewed and adjusted as better agronomic nutrient-content data become available.



In [1]:
from pathlib import Path
import re
import json
import unicodedata
from typing import Dict, Tuple

import numpy as np
import pandas as pd
import yaml
from IPython.display import display

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / 'inputs').exists() and (PROJECT_DIR.parent / 'inputs').exists():
    PROJECT_DIR = PROJECT_DIR.parent

DEFAULT_SUMMARY_STRATEGY = 'province'  # fallback only when metadata is missing
LATEST_FILTERED_SUMMARY_META_PATH = PROJECT_DIR / 'outputs/02_latest_filtered_export_summary.json'

summary_meta = {}
if LATEST_FILTERED_SUMMARY_META_PATH.exists():
    try:
        summary_meta = json.loads(LATEST_FILTERED_SUMMARY_META_PATH.read_text(encoding='utf-8')) or {}
    except Exception:
        summary_meta = {}

_meta_summary = str(summary_meta.get('summary_level', '')).strip()
if _meta_summary:
    SUMMARY_STRATEGY = _meta_summary
    SUMMARY_SOURCE = 'metadata'
elif available_strategy_inputs:
    m_latest = re.search(r'__summary_([a-z0-9_]+)$', available_strategy_inputs[0].stem)
    SUMMARY_STRATEGY = m_latest.group(1) if m_latest else DEFAULT_SUMMARY_STRATEGY
    SUMMARY_SOURCE = 'latest_filtered_csv'
else:
    SUMMARY_STRATEGY = DEFAULT_SUMMARY_STRATEGY
    SUMMARY_SOURCE = 'default'

SUMMARY_TOKEN = re.sub(r"[^a-z0-9]+", "_", str(SUMMARY_STRATEGY).strip().lower()).strip("_") or 'province'

preferred_input_csv = PROJECT_DIR / f'outputs/CSVs/02_espac_crop_lci_table_filtered__summary_{SUMMARY_TOKEN}.csv'
preferred_input_unc_csv = PROJECT_DIR / f'outputs/CSVs/02_espac_crop_lci_table_filtered__summary_{SUMMARY_TOKEN}_uncertainty.csv'
available_strategy_inputs = sorted((PROJECT_DIR / 'outputs/CSVs').glob('02_espac_crop_lci_table_filtered__summary_*.csv'), key=lambda p: p.stat().st_mtime, reverse=True)

INPUT_CSV = preferred_input_csv
INPUT_UNCERTAINTY_CSV = preferred_input_unc_csv
OUTPUT_CSV = PROJECT_DIR / f'outputs/CSVs/03-05_espac_crop_lci_table_filtered_dfe__summary_{SUMMARY_TOKEN}.csv'
OUTPUT_UNCERTAINTY_CSV = OUTPUT_CSV.with_name(f"{OUTPUT_CSV.stem}_uncertainty.csv")

FACTORS_YML = PROJECT_DIR / 'inputs/03-05_dfe_factors.yml'
FACTORS_XLSX = PROJECT_DIR / 'inputs/03_Factors.xlsx'
NUTRIENT_CONTENTS_YML = PROJECT_DIR / 'inputs/03_dfe_nutrient_contents.yml'
REGENERATE_YAML_FROM_XLSX = False

assert INPUT_CSV.exists(), (
    f'Missing inherited strategy input CSV: {INPUT_CSV}. '
    f'Latest metadata: {LATEST_FILTERED_SUMMARY_META_PATH if LATEST_FILTERED_SUMMARY_META_PATH.exists() else "(missing)"}. '
    f'Available summary files: {[p.name for p in available_strategy_inputs]}'
)
assert INPUT_UNCERTAINTY_CSV.exists(), (
    f'Missing inherited strategy uncertainty CSV: {INPUT_UNCERTAINTY_CSV}. '
    f'Expected from inherited summary strategy: {SUMMARY_STRATEGY}'
)

assert FACTORS_YML.exists() or FACTORS_XLSX.exists(), 'Need inputs/03-05_dfe_factors.yml or inputs/03_Factors.xlsx'
assert NUTRIENT_CONTENTS_YML.exists(), f'Missing nutrient-content YAML: {NUTRIENT_CONTENTS_YML}'

print(f'Inherited summary strategy: {SUMMARY_STRATEGY} (source: {SUMMARY_SOURCE})')
print(f'Metadata file: {LATEST_FILTERED_SUMMARY_META_PATH if LATEST_FILTERED_SUMMARY_META_PATH.exists() else "(missing)"}')
print(f'Using inherited strategy input CSV: {INPUT_CSV}')
print(f'Using inherited strategy uncertainty CSV: {INPUT_UNCERTAINTY_CSV}')










Inherited summary strategy: crop_group_national (source: metadata)
Metadata file: c:\Users\AAVADI\OneDrive - IRTA\Documents\_Proyectos\ESPAC\outputs\02_latest_filtered_export_summary.json
Using inherited strategy input CSV: c:\Users\AAVADI\OneDrive - IRTA\Documents\_Proyectos\ESPAC\outputs\CSVs\02_espac_crop_lci_table_filtered__summary_crop_group_national.csv
Using inherited strategy uncertainty CSV: c:\Users\AAVADI\OneDrive - IRTA\Documents\_Proyectos\ESPAC\outputs\CSVs\02_espac_crop_lci_table_filtered__summary_crop_group_national_uncertainty.csv


In [2]:
def export_factors_xlsx_to_yaml(xlsx_path: Path, yml_path: Path) -> None:
    import pandas as pd

    factors_df = pd.read_excel(xlsx_path, sheet_name='factors')
    refs_df = pd.read_excel(xlsx_path, sheet_name='references')
    factors_df = factors_df.rename(columns={c: str(c).strip() for c in factors_df.columns})

    factor_values = {}
    factor_details = {}
    for _, row in factors_df.iterrows():
        var = str(row.get('Var', '')).strip()
        if not var or var.lower() == 'nan':
            continue
        val = float(row.get('Val', 0) or 0)
        factor_values[var] = val
        detail = {'value': val}
        if 'Comment' in factors_df.columns and pd.notna(row.get('Comment')):
            detail['comment'] = str(row.get('Comment'))
        if 'Current value' in factors_df.columns and pd.notna(row.get('Current value')):
            detail['current_value'] = str(row.get('Current value'))
        factor_details[var] = detail

    if yml_path.exists():
        with yml_path.open('r', encoding='utf-8') as f:
            existing = yaml.safe_load(f) or {}
    else:
        existing = {}

    existing.setdefault('meta', {})
    existing['meta']['source_excel'] = str(xlsx_path).replace('\\', '/')
    existing['meta']['source_sheet'] = 'factors'
    existing['knime_factors'] = factor_values
    existing['knime_factor_details'] = factor_details
    existing['references_sheet_preview'] = refs_df.fillna('').astype(str).head(10).to_dict(orient='records')

    with yml_path.open('w', encoding='utf-8') as f:
        yaml.safe_dump(existing, f, sort_keys=False, allow_unicode=True)


if REGENERATE_YAML_FROM_XLSX:
    export_factors_xlsx_to_yaml(FACTORS_XLSX, FACTORS_YML)
    print(f'Regenerated {FACTORS_YML} from {FACTORS_XLSX}')

with FACTORS_YML.open('r', encoding='utf-8') as f:
    CFG = yaml.safe_load(f) or {}

KNIME_FACTORS = CFG.get('knime_factors', {})
ASSUMPTIONS = CFG.get('assumptions', {})
print('Loaded factors:', len(KNIME_FACTORS), 'from', FACTORS_YML)

with NUTRIENT_CONTENTS_YML.open('r', encoding='utf-8') as f:
    NUTRIENT_CFG = yaml.safe_load(f) or {}

NUTRIENT_MAPS = (NUTRIENT_CFG.get('espac_default_mappings', {}) or {})
print('Loaded nutrient-content mappings from', NUTRIENT_CONTENTS_YML)


Loaded factors: 52 from c:\Users\AAVADI\OneDrive - IRTA\Documents\_Proyectos\ESPAC\inputs\03-05_dfe_factors.yml
Loaded nutrient-content mappings from c:\Users\AAVADI\OneDrive - IRTA\Documents\_Proyectos\ESPAC\inputs\03_dfe_nutrient_contents.yml


In [3]:
def _norm_text(x: str) -> str:
    s = str(x).strip().upper()
    s = unicodedata.normalize('NFKD', s)
    s = ''.join(ch for ch in s if not unicodedata.combining(ch))
    s = re.sub(r'\s+', ' ', s)
    return s


def to_num(series: pd.Series) -> pd.Series:
    return pd.to_numeric(series, errors='coerce')


def get_factor(name: str, default: float = 0.0) -> float:
    try:
        return float(KNIME_FACTORS.get(name, default))
    except Exception:
        return float(default)


def infer_crop_type(crop: str, category: str) -> str:
    txt = _norm_text(crop)
    groups = (ASSUMPTIONS.get('crop_type_inference', {}) or {}).get('keyword_groups', {}) or {}
    for crop_type, keywords in groups.items():
        for kw in keywords or []:
            if _norm_text(kw) in txt:
                return crop_type
    fallback = ((ASSUMPTIONS.get('crop_type_inference', {}) or {}).get('fallback_by_category', {}) or {})
    return str(fallback.get(str(category).strip().lower(), 'cereal'))


def ensure_numeric_columns(df: pd.DataFrame, columns) -> pd.DataFrame:
    out = df.copy()
    for c in columns:
        if c not in out.columns:
            out[c] = 0.0
        out[c] = to_num(out[c]).fillna(0.0)
    return out


def apply_fraction_columns(df: pd.DataFrame, source_col: str, frac_cfg: Dict[str, float], prefix: str):
    amount = to_num(df.get(source_col, 0)).fillna(0.0)
    n = amount * float(frac_cfg.get('N', 0.0))
    p = amount * float(frac_cfg.get('P', 0.0))
    k = amount * float(frac_cfg.get('K', 0.0))
    return {
        f'{prefix}_N': n,
        f'{prefix}_P': p,
        f'{prefix}_K': k,
    }


def build_min_max_scenarios(df_mid: pd.DataFrame, df_unc: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    df_min = df_mid.copy()
    df_max = df_mid.copy()
    for c in df_mid.columns:
        cmin = f'{c}__minValue'
        cmax = f'{c}__maxValue'
        if cmin in df_unc.columns:
            df_min[c] = pd.to_numeric(df_unc[cmin], errors='coerce').fillna(pd.to_numeric(df_min[c], errors='coerce'))
        if cmax in df_unc.columns:
            df_max[c] = pd.to_numeric(df_unc[cmax], errors='coerce').fillna(pd.to_numeric(df_max[c], errors='coerce'))
    return df_min, df_max


def build_uncertainty_output(df_out: pd.DataFrame, df_unc_in: pd.DataFrame, df_min_out: pd.DataFrame, df_max_out: pd.DataFrame) -> pd.DataFrame:
    key_cols = [c for c in ['Region', 'Province', 'Crop', 'Category'] if c in df_out.columns]
    unc_base = df_out[key_cols].copy() if key_cols else pd.DataFrame(index=df_out.index)
    unc_data = {}

    for c in df_out.columns:
        if not pd.api.types.is_numeric_dtype(df_out[c]):
            continue
        cmin = f'{c}__minValue'
        cmax = f'{c}__maxValue'

        if cmin in df_unc_in.columns and cmax in df_unc_in.columns:
            lo = pd.to_numeric(df_unc_in[cmin], errors='coerce')
            hi = pd.to_numeric(df_unc_in[cmax], errors='coerce')
        else:
            lo = pd.to_numeric(df_min_out.get(c, df_out[c]), errors='coerce')
            hi = pd.to_numeric(df_max_out.get(c, df_out[c]), errors='coerce')

        mid = pd.to_numeric(df_out[c], errors='coerce')
        lo2 = np.minimum(np.minimum(lo, hi), mid)
        hi2 = np.maximum(np.maximum(lo, hi), mid)
        unc_data[cmin] = pd.Series(lo2, index=df_out.index).fillna(mid)
        unc_data[cmax] = pd.Series(hi2, index=df_out.index).fillna(mid)

    unc_values = pd.DataFrame(unc_data, index=df_out.index)
    return pd.concat([unc_base, unc_values], axis=1).copy()


In [4]:
def compute_dfe_columns(df_in: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    df = df_in.copy()

    # Expected source columns (missing ones are injected as zeros).
    source_numeric_cols = [
        'Yield_kgha', 'Seed_kgha',
        'Urea_kgha', 'Limestone_kgha', 'NPK_kgha', 'MAP_kgha', 'UAN_kgha', 'AN_kgha', 'AP_kgha', 'AS_kgha', 'CAN_kgha',
        'Organic_estiercol_kgha', 'Organic_fermentado_kgha', 'Organic_liquido_kgha', 'Total_fert_org_kgha',
        'Straw_kgha', 'Fuel_ha',
    ]
    df = ensure_numeric_columns(df, source_numeric_cols)

    if 'Crop' not in df.columns:
        df['Crop'] = ''
    if 'Category' not in df.columns:
        df['Category'] = ''

    # Crop type for residue-N defaults (KNIME script uses Custom_classification_3).
    df['crop_type_dfe'] = [infer_crop_type(c, cat) for c, cat in zip(df['Crop'], df['Category'])]

    crop_res_cfg = ASSUMPTIONS.get('crop_residues', {}) or {}
    n_content_by_crop_type = crop_res_cfg.get('n_content_by_crop_type', {}) or {}
    export_straw_default = float(crop_res_cfg.get('default_export_straw_fraction', 0.0) or 0.0)
    infer_straw = bool(crop_res_cfg.get('infer_straw_from_yield_if_missing', False))
    straw_ratio_map = crop_res_cfg.get('straw_to_yield_ratio_by_crop_type', {}) or {}

    straw = df['Straw_kgha'].copy()
    if infer_straw:
        inferred = pd.Series([
            float(straw_ratio_map.get(ct, 0.0)) for ct in df['crop_type_dfe']
        ], index=df.index) * df['Yield_kgha']
        straw = straw.where(straw > 0, inferred)
    export_straw = pd.Series(export_straw_default, index=df.index, dtype='float64')
    n_content_res = pd.Series([
        float(n_content_by_crop_type.get(ct, n_content_by_crop_type.get('cereal', 0.28 / 100)))
        for ct in df['crop_type_dfe']
    ], index=df.index, dtype='float64')
    N_crop_res = straw * (1 - export_straw.clip(lower=0, upper=1)) * n_content_res

    # Reconstruct nutrient totals using nutrient-content YAML (preferred), with fallback to legacy assumptions.
    fert_fracs = (NUTRIENT_MAPS.get('mineral_fertilizer_nutrient_fractions', {}) or ASSUMPTIONS.get('fertilizer_nutrient_fractions', {}) or {})
    org_fracs = (NUTRIENT_MAPS.get('organic_nutrient_fractions', {}) or ASSUMPTIONS.get('organic_nutrient_fractions', {}) or {})

    mineral_cols = ['Urea_kgha', 'Limestone_kgha', 'NPK_kgha', 'MAP_kgha', 'UAN_kgha', 'AN_kgha', 'AP_kgha', 'AS_kgha', 'CAN_kgha']
    min_N = pd.Series(0.0, index=df.index)
    min_P = pd.Series(0.0, index=df.index)
    min_K = pd.Series(0.0, index=df.index)
    for col in mineral_cols:
        fr = fert_fracs.get(col, {}) or {}
        min_N += df[col] * float(fr.get('N', 0.0))
        min_P += df[col] * float(fr.get('P', 0.0))
        min_K += df[col] * float(fr.get('K', 0.0))

    org_N = pd.Series(0.0, index=df.index)
    org_P = pd.Series(0.0, index=df.index)
    org_K = pd.Series(0.0, index=df.index)
    org_P_solid = pd.Series(0.0, index=df.index)
    org_P_liquid = pd.Series(0.0, index=df.index)
    for col, cfg in org_fracs.items():
        if col not in df.columns:
            df[col] = 0.0
        amount = df[col]
        n_part = amount * float((cfg or {}).get('N', 0.0))
        p_part = amount * float((cfg or {}).get('P', 0.0))
        k_part = amount * float((cfg or {}).get('K', 0.0))
        org_N += n_part
        org_P += p_part
        org_K += k_part
        phase = str((cfg or {}).get('phase', '')).strip().lower()
        if phase == 'solid':
            org_P_solid += p_part
        elif phase == 'liquid':
            org_P_liquid += p_part

    # KNIME-aligned derived nutrient totals
    df['N_min'] = min_N
    df['P_min'] = min_P
    df['K_min'] = min_K
    df['N_orga'] = org_N
    df['P_orga'] = org_P
    df['K_orga'] = org_K
    df['N_tot'] = df['N_min'] + df['N_orga']
    df['P_tot'] = df['P_min'] + df['P_orga']
    df['K_tot'] = df['K_min'] + df['K_orga']

    # KNIME script factors (direct field emissions block)
    EF_NH3_u = get_factor('EF_NH3_u')
    CF_NH3_u = get_factor('EF_NH3_u')  # KNIME script uses EF_NH3_u here (likely typo); preserved intentionally.
    EF_NH3_m = get_factor('EF_NH3_m')
    CF_NH3_m = get_factor('CF_NH3_m')
    FracLEACH = get_factor('FracLEACH')
    FracGASF = get_factor('FracGASF')
    FracGASM = get_factor('FracGASM')
    EF_N2O_d = get_factor('EF_N2O_d')
    EF_N2O_a = get_factor('EF_N2O_a')
    EF_N2O_l = get_factor('EF_N2O_l')
    EF_NOx_effective = FracGASM  # KNIME script sets EF_NOx = FracGASM.
    EF_limestone = get_factor('EF_limestone')
    EF_urea = get_factor('EF_urea')
    RUSLE_R = get_factor('RUSLE_R')
    RUSLE_K = get_factor('RUSLE_K')
    RUSLE_LS = get_factor('RUSLE_LS')
    RUSLE_C = get_factor('RUSLE_C')
    RUSLE_P = get_factor('RUSLE_P')
    Psoil = get_factor('Psoil')
    e = get_factor('e')
    r = get_factor('r')
    t = get_factor('t')
    kl = get_factor('kl')
    kr = get_factor('kr')
    ks1 = get_factor('ks1')
    ks2 = get_factor('ks2')
    kn = get_factor('kn')
    kt = get_factor('kt')
    s = get_factor('s')
    kd = get_factor('kd')

    N_applied = df['N_tot']
    P_applied = df['P_tot']
    _ = P_applied  # retained for traceability to KNIME variable names

    # Direct emissions from fertilisers (KNIME formulas)
    df['kgNH3N_ha'] = ((df['N_min'] * EF_NH3_u * CF_NH3_u) + (df['N_orga'] * EF_NH3_m * CF_NH3_m)) * 14 / 17
    df['kgNOxN_ha'] = (N_applied - df['kgNH3N_ha']) * EF_NOx_effective * 14 / 46
    df['kgNO3N_ha'] = N_applied * FracLEACH
    df['kgN2ON_ha_direct'] = (N_applied + N_crop_res) * EF_N2O_d
    df['kgN2ON_ha_indirect'] = (df['N_min'] * FracGASF + df['N_orga'] * FracGASM) * EF_N2O_a + (df['kgNO3N_ha'] * EF_N2O_l)
    df['kgN2ON_ha'] = df['kgN2ON_ha_direct'] + df['kgN2ON_ha_indirect']
    df['kgCO2urea_ha'] = df['Urea_kgha'] * EF_urea
    df['kgCO2limestone_ha'] = df['Limestone_kgha'] * EF_limestone
    df['kgCO2_fert_ha'] = df['kgCO2urea_ha'] + df['kgCO2limestone_ha']

    soil_erosion_tha = RUSLE_R * RUSLE_K * RUSLE_LS * RUSLE_C * RUSLE_P
    df['soil_erosion_tha'] = soil_erosion_tha

    fm = df['P_min']
    fos = org_P_solid
    fol = org_P_liquid
    df['kgP_ha_erosion'] = soil_erosion_tha * Psoil * e * r
    df['kgP_ha_runoff'] = s * kr * ks1 * kn * kt * (1 + 0.2 / 80 * fm + 0.7 / 80 * fol + 0.4 / 80 * fos)
    df['kgP_ha_leaching'] = kl * ks2 * kn * (1 + 0.2 / 80 * fol)
    df['kgP_ha_drainage'] = df['kgP_ha_leaching'] * kd
    df['kgP_total'] = df['kgP_ha_erosion'] + df['kgP_ha_runoff'] + df['kgP_ha_leaching'] + df['kgP_ha_drainage']

    # Trace elements: compute row-level metals from organic fertiliser amounts when present,
    # using PRO-derived median metal contents from inputs/03_dfe_nutrient_contents.yml (mg/kg product, wet-matter basis).
    # Fallback to legacy constants from inputs/03-05_dfe_factors.yml for rows without organic inputs.
    trace_metals = ['Cd', 'Cu', 'Zn', 'Pb', 'Ni', 'Cr', 'Hg']
    organic_trace_cfg = (NUTRIENT_MAPS.get('organic_trace_metal_factors_mg_per_kg', {}) or {})
    organic_trace_basis_mass = pd.Series(0.0, index=df.index, dtype='float64')
    trace_from_organic = {m: pd.Series(0.0, index=df.index, dtype='float64') for m in trace_metals}

    for org_col, metal_map in organic_trace_cfg.items():
        if org_col not in df.columns:
            continue
        amount = to_num(df[org_col]).fillna(0.0)
        organic_trace_basis_mass = organic_trace_basis_mass + amount
        for metal in trace_metals:
            mg_per_kg = float((metal_map or {}).get(metal, 0.0) or 0.0)
            trace_from_organic[metal] = trace_from_organic[metal] + amount * mg_per_kg * 1e-6

    for metal in trace_metals:
        # Data-driven policy: emit trace metals only when organic fertiliser input is present.
        df[f'{metal}_kg_ha'] = np.where(organic_trace_basis_mass > 0, trace_from_organic[metal], 0.0)

    df['trace_metals_from_organic'] = organic_trace_basis_mass > 0

    # Helpful diagnostics on reconstructed inputs
    diagnostics = pd.DataFrame({
        'metric': [
            'rows',
            'rows_with_positive_N_tot',
            'rows_with_positive_P_tot',
            'rows_with_positive_seed',
            'rows_with_positive_dfe_N2O',
        ],
        'value': [
            len(df),
            int((df['N_tot'] > 0).sum()),
            int((df['P_tot'] > 0).sum()),
            int((df['Seed_kgha'] > 0).sum()) if 'Seed_kgha' in df.columns else 0,
            int((df['kgN2ON_ha'] > 0).sum()),
        ]
    })

    return df, diagnostics


### Uncertainty Processing (new)
This notebook now propagates uncertainty through DFE formulas.
- It reads `outputs/CSVs/02_espac_crop_lci_table_filtered__summary_<strategy>_uncertainty.csv` when available.
- It computes DFE with min-input and max-input scenarios, then writes bounds as `__minValue` / `__maxValue`.
- It exports `outputs/CSVs/03-05_espac_crop_lci_table_filtered_dfe__summary_<strategy>_uncertainty.csv` for XML generation.



In [5]:
df_in = pd.read_csv(INPUT_CSV)
print('Input shape:', df_in.shape)

if INPUT_UNCERTAINTY_CSV.exists():
    df_unc_in = pd.read_csv(INPUT_UNCERTAINTY_CSV)
    print('Input uncertainty shape:', df_unc_in.shape)
else:
    df_unc_in = pd.DataFrame(index=df_in.index)
    print(f'Input uncertainty CSV not found: {INPUT_UNCERTAINTY_CSV}. Using point values as min/max fallback.')

# Preserve original column order and append DFE columns at the end.
df_out, dfe_diag = compute_dfe_columns(df_in)

# Build min/max DFE scenarios from uncertainty input bounds.
df_min_in, df_max_in = build_min_max_scenarios(df_in, df_unc_in)
df_min_out, _ = compute_dfe_columns(df_min_in)
df_max_out, _ = compute_dfe_columns(df_max_in)

df_unc_out = build_uncertainty_output(df_out, df_unc_in, df_min_out, df_max_out)

# Drop helper source columns that were injected as zero placeholders only because they were absent in the input.
synthetic_source_helpers = ['Urea_kgha', 'Limestone_kgha', 'MAP_kgha', 'UAN_kgha', 'CAN_kgha', 'Straw_kgha']
for c in synthetic_source_helpers:
    if c not in df_in.columns and c in df_out.columns:
        df_out = df_out.drop(columns=[c])
        cmin = f'{c}__minValue'
        cmax = f'{c}__maxValue'
        df_unc_out = df_unc_out.drop(columns=[x for x in [cmin, cmax] if x in df_unc_out.columns], errors='ignore')

new_cols = [c for c in df_out.columns if c not in df_in.columns]
print('New columns added:', len(new_cols))
print(new_cols)

# Optional clean-up: reorder so original columns stay first.
df_out = df_out[list(df_in.columns) + new_cols]

# Write outputs
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
df_out.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')
print(f'Saved DFE-augmented table to: {OUTPUT_CSV}')
print('Output shape:', df_out.shape)

df_unc_out.to_csv(OUTPUT_UNCERTAINTY_CSV, index=False, encoding='utf-8-sig')
print(f'Saved DFE uncertainty table to: {OUTPUT_UNCERTAINTY_CSV}')

# Quick summaries
num_show = [
    'N_tot', 'P_tot', 'K_tot',
    'kgNH3N_ha', 'kgNOxN_ha', 'kgNO3N_ha', 'kgN2ON_ha',
    'kgCO2_fert_ha', 'kgP_total'
]
num_show = [c for c in num_show if c in df_out.columns]
if num_show:
    display(df_out[['Crop', 'Category'] + num_show].head(15))
    display(df_out[num_show].describe().T)

display(dfe_diag)


Input shape: (10, 32)
Input uncertainty shape: (10, 88)
New columns added: 33
['crop_type_dfe', 'N_min', 'P_min', 'K_min', 'N_orga', 'P_orga', 'K_orga', 'N_tot', 'P_tot', 'K_tot', 'kgNH3N_ha', 'kgNOxN_ha', 'kgNO3N_ha', 'kgN2ON_ha_direct', 'kgN2ON_ha_indirect', 'kgN2ON_ha', 'kgCO2urea_ha', 'kgCO2limestone_ha', 'kgCO2_fert_ha', 'soil_erosion_tha', 'kgP_ha_erosion', 'kgP_ha_runoff', 'kgP_ha_leaching', 'kgP_ha_drainage', 'kgP_total', 'Cd_kg_ha', 'Cu_kg_ha', 'Zn_kg_ha', 'Pb_kg_ha', 'Ni_kg_ha', 'Cr_kg_ha', 'Hg_kg_ha', 'trace_metals_from_organic']
Saved DFE-augmented table to: c:\Users\AAVADI\OneDrive - IRTA\Documents\_Proyectos\ESPAC\outputs\CSVs\03-05_espac_crop_lci_table_filtered_dfe__summary_crop_group_national.csv
Output shape: (10, 65)
Saved DFE uncertainty table to: c:\Users\AAVADI\OneDrive - IRTA\Documents\_Proyectos\ESPAC\outputs\CSVs\03-05_espac_crop_lci_table_filtered_dfe__summary_crop_group_national_uncertainty.csv


,Crop,Category,N_tot,P_tot,K_tot,kgNH3N_ha,kgNOxN_ha,kgNO3N_ha,kgN2ON_ha,kgCO2_fert_ha,kgP_total
0,(all crops in group),transitory,85.467649,34.397758,84.915297,2.423087,4.802142,26.494971,0.780746,0.0,1.725654
1,(all crops in group),permanent,79.462948,30.566687,81.096671,1.965446,4.481377,24.633514,0.719686,0.0,1.723115
2,(all crops in group),permanent,71.213677,26.432662,74.263244,1.674744,4.021164,22.076240,0.643102,0.0,1.720840
3,(all crops in group),transitory,67.188737,96.306141,89.351398,1.870498,3.777098,20.828508,0.613026,0.0,1.752340
4,(all crops in group),permanent,90.602023,30.366692,101.078148,2.745715,5.080387,28.086627,0.831472,0.0,1.723676
5,(all crops in group),transitory,141.567922,49.510309,144.908518,6.211077,7.827157,43.886056,1.340679,0.0,1.729021
6,(all crops in group),transitory,66.256649,32.857033,59.814806,2.136923,3.707793,20.539561,0.610836,0.0,1.724664
7,(all crops in group),transitory,142.735864,35.623101,149.430470,4.581977,7.988899,44.248118,1.315450,0.0,1.725188
8,(all crops in group),permanent,169.641245,41.015717,621.199993,5.511938,9.490956,52.588786,1.564841,0.0,1.733400
9,(all crops in group),transitory,168.399673,49.895400,168.785995,6.434465,9.365814,52.203899,1.574182,0.0,1.727677


,count,mean,std,min,25%,50%,75%,max
N_tot,10.0,108.253639,42.382604,66.256649,73.275995,88.034836,142.443879,169.641245
P_tot,10.0,42.697150,20.424541,26.432662,31.139274,35.010429,47.386661,96.306141
K_tot,10.0,157.484454,166.924180,59.814806,82.051327,95.214773,148.299982,621.199993
kgNH3N_ha,10.0,3.555587,1.917088,1.674744,2.008315,2.584401,5.279448,6.434465
kgNOxN_ha,10.0,6.054279,2.344694,3.707793,4.136218,4.941264,7.948463,9.490956
kgNO3N_ha,10.0,33.558628,13.138607,20.539561,22.715558,27.290799,44.157602,52.588786
kgN2ON_ha,10.0,0.999402,0.401099,0.610836,0.662248,0.806109,1.334372,1.574182
kgCO2_fert_ha,10.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
kgP_total,10.0,1.728558,0.009055,1.720840,1.723923,1.725421,1.728685,1.752340


,metric,value
0,rows,10
1,rows_with_positive_N_tot,10
2,rows_with_positive_P_tot,10
3,rows_with_positive_seed,8
4,rows_with_positive_dfe_N2O,10


## Next steps

- Review and calibrate nutrient-content assumptions in `inputs/03-05_dfe_factors.yml`.
- If you want tighter alignment with the KNIME workflow, add explicit nutrient totals (`N_tot`, `P_tot`, `K_tot`) upstream in the ESPAC LCI table and let this notebook use them directly.
